# **ST 554 Final Project**
---
Authored by: Jamie Loring

Collaborator: Dr. Justin Post (code from lecture videos & slides)

## **Setup**

The code below imports the required modules that are needed for this project.

In [1]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, Binarizer, VectorAssembler, OneHotEncoder, PCA, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col
from pyspark.ml import Pipeline

The code below sets up our `spark` object and only allows errors to print out in the future (i.e., it suppresses warnings).

In [2]:
spark = SparkSession.builder.master('local[*]').appName('my_app').getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/28 15:28:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/28 15:28:46 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/28 15:28:46 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/28 15:28:46 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/28 15:28:46 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


The code below reads in the `power_ml_data.csv` file from the provided URL using `pandas`.

In [3]:
power_data = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
power_data.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


Now, we convert our `pandas dataframe` to a `spark` SQL style dataframe. This new object will be called `power_ML`. For better readability, throughout this project, I will display results using `.toPandas()` and `.head()` rather than `.show()`. It is important to note that this choice does not change `power_ML` back to a `pandas` or `pandas-on-spark` dataframe; rather, it just changes the way it is *displayed*.

In [5]:
power_ML = spark.createDataFrame(power_data)
power_ML.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


We are going to treat the `Power_Zone_3` variable as our response variable. We want to imagine that we know that the `Power_Zone_3` reading is going to go offline in the future, and we need to be able to predict that value appropriately using the other variables in the dataset.

## **Fitting the Model**

We are going to fit an elastic net model using CV without a training & testing split. Before we can fit our model, we have to perform the required transformations using `MLlib` functions. All (nested) transformations will be saved as objects so 1) We can display our progress through the transformations and 2) We can check to make sure we do not miss any transformations in the final pipeline.

First, we need to check to see how the `Hour` column is stored. 

In [6]:
power_ML.dtypes

[('Temperature', 'double'),
 ('Humidity', 'double'),
 ('Wind_Speed', 'double'),
 ('General_Diffuse_Flows', 'double'),
 ('Diffuse_Flows', 'double'),
 ('Power_Zone_1', 'double'),
 ('Power_Zone_2', 'double'),
 ('Power_Zone_3', 'double'),
 ('Month', 'bigint'),
 ('Hour', 'bigint')]

Now that we have confirmed that it's not a `DoubleType`, we will need to use an SQL transformer to cast the `Hour` variable as a `DoubleType`.

In [7]:
cast_hour = SQLTransformer(
    statement = """
                SELECT *, CAST(Hour as DOUBLE) as Hour_Double_Type FROM __THIS__
                """
            )

tf_1 = cast_hour.transform(power_ML)
tf_1.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,Hour_Double_Type
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,0.0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,0.0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,0.0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,0.0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,0.0


Next, we need to binarize the updated `Hour_Double_Type` column based on it being less than 6.5 or not. We are essentially creating a night vs. day variable.

In [8]:
binarize_hour = Binarizer(threshold = 6.5, inputCol = "Hour_Double_Type", outputCol = "Night_vs_Day")

tf_2 = binarize_hour.transform(tf_1)
tf_2.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,Hour_Double_Type,Night_vs_Day
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,0.0,0.0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,0.0,0.0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,0.0,0.0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,0.0,0.0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,0.0,0.0


Continuing on, we need to one-hot encode the `Month` variable. Since `Month` is already an integer, we do _**not**_ need to use `StringIndexer()`.

In [10]:
encoder_OHE = OneHotEncoder(inputCols = ["Month"], outputCols = ["Month_OHE"])
model_OHE = encoder_OHE.fit(tf_2)

tf_3 = model_OHE.transform(tf_2)
tf_3.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,Hour_Double_Type,Night_vs_Day,Month_OHE
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


Our next step is to run a PCA fit on the following variables:
- `Temperature`
- `Humidity`
- `Wind_Speed`
- `General_Diffuse_Flows`
- `Diffuse_Flows`

We will also need to standarize our predictors using `StandardScaler()`.

**Note:** This will be a multi-step process, outlined with in-line comments in the code chunk below.

In [12]:
# use VectorAssembler() to place the above variables in a column together for use with the PCA() estimator
assembler_PCA = VectorAssembler(
                inputCols = ["Temperature", "Humidity", "Wind_Speed",
                             "General_Diffuse_Flows", "Diffuse_Flows"],
                outputCol = "features_for_PCA",
                handleInvalid = "keep"
             )

# update transformations
tf_4 = assembler_PCA.transform(tf_3)

# use StandardScaler() to standardize predictors
scaler_PCA = StandardScaler(inputCol = "features_for_PCA", outputCol = "features_for_PCA_scaled",
                            withStd = True, withMean = True)
# fit scaler_PCA
scaler_fit = scaler_PCA.fit(tf_4)

# update transformations
tf_5 = scaler_fit.transform(tf_4)

# run PCA, using 2 PC's in the transformation
PCA_func = PCA(k = 2, inputCol = "features_for_PCA_scaled", outputCol = "PCA_features_scaled")

# fit the PCA model
PCA_model = PCA_func.fit(tf_5)

# update transformations
tf_6 = PCA_model.transform(tf_5)

# show updated transformations
tf_6.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,Hour_Double_Type,Night_vs_Day,Month_OHE,features_for_PCA,features_for_PCA_scaled,PCA_features_scaled
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.559, 73.8, 0.083, 0.051, 0.119]","[-2.1079477438809433, 0.3542085264292604, -0.7...","[2.069074321346372, 0.8078678829472259]"
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.414, 74.5, 0.083, 0.07, 0.085]","[-2.1328903699941857, 0.3991947174962608, -0.7...","[2.102921063880654, 0.8152476222806389]"
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.313, 74.5, 0.08, 0.062, 0.1]","[-2.1502641992178924, 0.3991947174962608, -0.8...","[2.112066263379101, 0.8227151924829956]"
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.121, 75.0, 0.083, 0.091, 0.096]","[-2.183291676554047, 0.4313277111155467, -0.79...","[2.1435031847422197, 0.8329135817940964]"
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[5.921, 75.7, 0.081, 0.048, 0.085]","[-2.217695298779209, 0.47631390218254716, -0.8...","[2.1824440036627006, 0.8444681624812329]"


We now need to put our model predictors into a single column called features. We can do this with `VectorAssembler()`. We will also use `StandardScaler()` to standardize our features. Since `PCA_features_scaled` has already been standardized, we do not need to re-standardize it, so this will be a multi-step process.

In [13]:
# assemble features vector without PCA for scaling
assembler_features = VectorAssembler(
                          inputCols = ["Night_vs_Day", "Power_Zone_1",
                                       "Power_Zone_2", "Month_OHE"],
                          outputCol = "features_pre_scale",
                          handleInvalid = "keep"
                    )

# update transformations
tf_7 = assembler_features.transform(tf_6)

# use StandardScaler() to standardize features -- remember PCA_features_scaled is already standardized!
scaler_features = StandardScaler(inputCol = "features_pre_scale", outputCol = "features_scaled",
                                 withStd = True, withMean = True)

# fit scaler_features
scaler_fit_feat = scaler_features.fit(tf_7)

# update transformations
tf_8 = scaler_fit_feat.transform(tf_7)

# use VectorAssembler() again to combine PCA_features_scaled and features_scaled into one features column
assembler_features_final = VectorAssembler(
                                inputCols = ["PCA_features_scaled", "features_scaled"],
                                outputCol = "features",
                                handleInvalid = "keep"
                        )

# update transformations
tf_9 = assembler_features_final.transform(tf_8)

# show updated transformations
tf_9.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,Hour_Double_Type,Night_vs_Day,Month_OHE,features_for_PCA,features_for_PCA_scaled,PCA_features_scaled,features_pre_scale,features_scaled,features
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.559, 73.8, 0.083, 0.051, 0.119]","[-2.1079477438809433, 0.3542085264292604, -0.7...","[2.069074321346372, 0.8078678829472259]","(0.0, 34055.6962, 16128.87538, 0.0, 1.0, 0.0, ...","[-1.5544690531019132, 0.24130775579578978, -0....","[2.069074321346372, 0.8078678829472259, -1.554..."
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.414, 74.5, 0.083, 0.07, 0.085]","[-2.1328903699941857, 0.3991947174962608, -0.7...","[2.102921063880654, 0.8152476222806389]","(0.0, 29814.68354, 19375.07599, 0.0, 1.0, 0.0,...","[-1.5544690531019132, -0.35350356900601565, -0...","[2.102921063880654, 0.8152476222806389, -1.554..."
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.313, 74.5, 0.08, 0.062, 0.1]","[-2.1502641992178924, 0.3991947174962608, -0.8...","[2.112066263379101, 0.8227151924829956]","(0.0, 29128.10127, 19006.68693, 0.0, 1.0, 0.0,...","[-1.5544690531019132, -0.449798237837335, -0.3...","[2.112066263379101, 0.8227151924829956, -1.554..."
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.121, 75.0, 0.083, 0.091, 0.096]","[-2.183291676554047, 0.4313277111155467, -0.79...","[2.1435031847422197, 0.8329135817940964]","(0.0, 28228.86076, 18361.09422, 0.0, 1.0, 0.0,...","[-1.5544690531019132, -0.5759186911227896, -0....","[2.1435031847422197, 0.8329135817940964, -1.55..."
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[5.921, 75.7, 0.081, 0.048, 0.085]","[-2.217695298779209, 0.47631390218254716, -0.8...","[2.1824440036627006, 0.8444681624812329]","(0.0, 27335.6962, 17872.34043, 0.0, 1.0, 0.0, ...","[-1.5544690531019132, -0.7011869790980542, -0....","[2.1824440036627006, 0.8444681624812329, -1.55..."


Finally, we rename the response variable, `Power_Zone_3`, as `label`. We will also use the SQL transformer to select the `features` vector that was created in the previous step.

In [14]:
label_and_features = SQLTransformer(
    statement = """
                SELECT features, Power_Zone_3 as label FROM __THIS__
                """
            )

# update transformations
tf_10 = label_and_features.transform(tf_9)
tf_10.toPandas().head()

,features,label
0,"[2.069074321346372, 0.8078678829472259, -1.554...",20240.96386
1,"[2.102921063880654, 0.8152476222806389, -1.554...",20131.08434
2,"[2.112066263379101, 0.8227151924829956, -1.554...",19668.43373
3,"[2.1435031847422197, 0.8329135817940964, -1.55...",18899.27711
4,"[2.1824440036627006, 0.8444681624812329, -1.55...",18442.40964


We can now put all of our transformations into the pipeline. We will have 10 of them, plus our linear regression instance. They are:
- `cast_hour`
- `binarize_hour`
- `encoder_OHE`
- `assembler_PCA`
- `scaler_PCA`
- `PCA_func`
- `assembler_features`
- `scaler_features`
- `assembler_features_final`
- `label_and_features`
- and `lr`, which will be the instance of our `LinearRegression()`

**Note:** Only the transformers and estimators need to be put into the pipeline since the pipeline will automatically handle which ones need to be fit. The fit objects up to this point were only created to show the progress of the transformations and check their functionality, but they themselves do not go into the pipeline!

*Example:* `encoder_OHE` *goes into the pipeline rather than* `model_OHE`

In [15]:
lr = LinearRegression()
pipeline = Pipeline(stages = [cast_hour, binarize_hour, encoder_OHE, assembler_PCA,
                              scaler_PCA, PCA_func, assembler_features, scaler_features,
                              assembler_features_final, label_and_features, lr])

We are now ready to fit our elastic net model. We are provided with the following grid for `regParam` and `elasticNetParam`:
- `regParam`: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
- `elasticNetParam`: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1

Since our `pipeline` object is already set up, we have the following remaining steps to complete in creating our model:
- build the parameter grid
- use cross-validation (5 folds) to choose the best combination of parameters using RMSE
- fit the model

**Note:** The use of `parallelism = 128` in `CrossValidator()` speeds up the runtime from about 12 minutes to about 6 minutes! Please be patient! Additionally, the red line that appears under the code chunk is *not* representative of an actual error, but rather staging outputs from the log.

In [16]:
# build parameter grid
paramGrid = ParamGridBuilder() \
            .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
            .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
            .build()


# set up cross validation with pipeline
crossval = CrossValidator(estimator = pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName = 'rmse'),
                          numFolds = 5,
                          seed = 44,
                          parallelism = 128)

# fit the model using cross-validation to choose the best model
cvModel = crossval.fit(power_ML)

Although the `.bestModel` attribute will store the best model as a result of cross-validation by default, we want to look to see what model is the best. The code below prints the RMSE for the model fit with its corresponding parameters, `regParam` and `elasticNetParam`. I then sort this output in ascending order so we can see the lowest RMSE first (along with its corresponding parameters).

In [17]:
my_list = []

for i in range(len(paramGrid)):
    my_list.append([cvModel.avgMetrics[i], paramGrid[i].values()])
    
my_list.sort(key=lambda x: x[0]) #specifies sorting by avgMetrics rather than the dict_values
my_list

[[2175.025614452874, dict_values([0.25, 0.5])],
 [2175.0335187645323, dict_values([0.5, 0.25])],
 [2175.03354546321, dict_values([0.5, 0.05])],
 [2175.0341016574057, dict_values([0.5, 0.1])],
 [2175.0349079585444, dict_values([0.05, 0.5])],
 [2175.035462853383, dict_values([0.25, 0.1])],
 [2175.035512165499, dict_values([0.1, 0.95])],
 [2175.035587292835, dict_values([0.05, 0.98])],
 [2175.0356041252753, dict_values([0.05, 0.9])],
 [2175.0356236594257, dict_values([0.05, 0.95])],
 [2175.035731007239, dict_values([0.05, 0.99])],
 [2175.035734665818, dict_values([0.05, 1.0])],
 [2175.035803359566, dict_values([0.05, 0.25])],
 [2175.035833255783, dict_values([0.1, 0.0])],
 [2175.035841619817, dict_values([0.1, 0.1])],
 [2175.0358417832153, dict_values([0.25, 0.0])],
 [2175.035842577646, dict_values([0.05, 0.75])],
 [2175.03585093693, dict_values([0.05, 0.0])],
 [2175.0358805904766, dict_values([0.0, 0.05])],
 [2175.035880590478, dict_values([0.0, 0.99])],
 [2175.0358805904784, dict_values

Thus, the best multiple linear regression model based on cross-validation with RMSE is one with a regularization parameter of 0.25 and an elastic net parameter of 0.5. This tells us that we have a combination of L1 and L2 penalties; hence, we are fitting an elastic net model! We can now print the intercept and coefficients of this best model, as well as the CV error (with its corresponding tuning parameters).

In [18]:
# need to use indexing since the model is the last stage of the pipeline
print('Intercept:', cvModel.bestModel.stages[-1].intercept)
print('Coefficients (in Features Order):', cvModel.bestModel.stages[-1].coefficients)

print(' ')

print("Best regParam:", cvModel.bestModel.stages[-1]._java_obj.getRegParam())
print("Best elasticNetParam:", cvModel.bestModel.stages[-1]._java_obj.getElasticNetParam())
print('Resulting Best CV RMSE:', round(min(cvModel.avgMetrics), 4))

Intercept: 17831.197607816746
Coefficients (in Features Order): [281.3375940841672,-508.5488523409968,-933.3422279345797,4380.973673101184,582.7712870595216,0.0,1670.643619500635,1521.0143011644466,1488.0485558207497,1932.6899471239403,1411.9841811335264,1784.2129277383071,3556.719789082987,2402.836302038812,373.5950605539663,-54.72243392986636,504.5217858788961]
 
Best regParam: 0.25
Best elasticNetParam: 0.5
Resulting Best CV RMSE: 2175.0256


We now report the training set RMSE, i.e., the RMSE on the full dataset, by using the fitted model as a transformer and evaluating on the entire training (full) dataset.

In [20]:
training_rmse = RegressionEvaluator(metricName = 'rmse').evaluate(cvModel.transform(power_ML))

print('Entire Training (Full) Dataset RMSE:', round(training_rmse, 5))

Entire Training (Full) Dataset RMSE: 2174.44967


The last step is to take the outputted transformations from the model (i.e., the predictions) and create a `residual` column defined as `label` - `prediction`. We will print out a resulting dataframe with the following columns:
- `residual`
- `label`
- `prediction`

In [21]:
# store previous model transformer in an object
tf_final = cvModel.transform(power_ML)

# create residual column
resids_label_preds = tf_final.withColumn("residual", col("label") - col("prediction"))
resids_label_preds.select("residual", "label", "prediction").toPandas()

,residual,label,prediction
0,-695.215712,20240.96386,20936.179572
1,1431.166976,20131.08434,18699.917364
2,1432.893079,19668.43373,18235.540651
3,1284.964277,18899.27711,17614.312833
4,1426.591996,18442.40964,17015.817644
...,...,...,...
47169,2469.850984,14780.31212,12310.461136
47170,2647.362365,14428.81152,11781.449155
47171,2633.732742,13806.48259,11172.749848
47172,2794.162807,13512.60504,10718.442233


## **Streaming**
In this part of the project, we will be using the model created in the previous section with streaming data!

### **Reading a Stream**
In my `jupyterhub` main directory, I have created a folder called `csv_files_final`. This folder will be used to read in the stream in the form of `.csv` files. 

The code below sets up the schema for the stream by using the schema from the original data as we did in the Homework 10 assignment.

In [22]:
myschema = power_ML.schema
power_stream = spark.readStream.option("header", "true").schema(myschema).csv("csv_files_final")

### **Transformations and Aggregations**

First, with the stream, we use the model transformer to obtain predictions from the incoming data. This can be accomplished by creating an object very similar to `tf_final`. The only difference is that rather than supplying `power_ML` -- our data object -- to the `transform()` method, we will supply the stream object -- `power_stream`. The newly created object will be called `power_transform`, and we can use this to create a `residual` column.

In [23]:
# use model transformer with stream
power_transform = cvModel.transform(power_stream)

# create residual column
pwr_tf_resids = power_transform.withColumn("residual", col("label") - col("prediction")) \
                               .select("residual", "label", "prediction")

Next, we can use the stream again, with another transformation on the original stream, by modifying the response variable (`Power_Zone_3`) to be called `label`. This will be done using the `.withColumnRenamed()` method, rather than an SQL Transformer, because we want to keep all the original variables as well.

In [24]:
label_rename = power_stream.withColumnRenamed("Power_Zone_3", "label")

We can now join the above previous transform with this stream based on the `label` variable. This will be done with an inner join via the `.join()` method.

In [25]:
stream_join = pwr_tf_resids.join(label_rename, "label", "inner")

### **Writing Step**
We are now ready to write the `stream_join`. We will write it to the console using the `append` output mode. The code below will start the query.

In [26]:
query = stream_join.writeStream.outputMode("append").format("console").start()

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|          residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|22295.27273|2801.9389026680947|19493.333827331906|      20.85|   54.88|     0.083|                0.088|        0.133| 32699.93541| 20349.89817|    4|  23|
| 10542.6506| 847.3996302289306|  9695.25096977107|      11.09|    71.1|      4.92|                0.048|        0.056| 21347.69231| 17397.52066|   11|   3|
|23856.19433| 828.3320738941911|23027.862256105807|      19.98|    85.2|     0.072|                0.062|        0.078

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|    19200.0| -2991.248863865454|22191.248863865454|      20.53|    88.1|     0.074|                 0.45|        0.415| 46729.10284| 29053.94191|   10|  18|
|13461.57303| 1289.2576433844843|12172.315386615515|      21.05|    79.1|     4.919|                0.062|        0.133| 26378.76106| 14785.44699|    9|   4|
|26657.30408|-1249.7572696032657|27907.061349603267|      26.75|    75.0|     4.905|                160.8|       

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|          residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|13329.45455|-1675.256400810391|15004.710950810391|      17.19|    64.7|     0.087|                286.5|        262.7| 27615.75888|   17699.389|    4|   8|
|6926.290516|1155.5779563309125| 5770.712559669088|      18.33|    50.1|     0.082|                0.048|        0.122|  22339.1635| 17987.11261|   12|   7|
|16903.71859| -601.181976725482|17504.900566725482|      12.08|    86.3|     4.917|                285.8|        281.2

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16579.69605|  392.6177803870942|16187.078269612904|       20.2|    78.9|     4.916|                0.059|        0.137| 37257.24289| 23187.13693|   10|  22|
|27653.81818|  948.6142797861721|26705.203900213826|      17.95|   69.64|     0.083|                0.062|        0.108| 43575.11302| 23993.89002|    4|  20|
|30306.27615| -631.5117662150042|30937.787916215006|      23.84|   67.86|     4.905|                0.084|       

-------------------------------------------
Batch: 4
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|11490.82067| 1738.6419381768483|  9752.17873182315|      18.59|    81.7|     4.915|                0.033|        0.148| 24804.55142| 15613.69295|   10|   3|
|19506.95925|-1874.3282498826156|21381.287499882616|      20.46|    76.9|     4.913|                0.073|        0.152| 28895.89345| 18513.19958|    8|   4|
|11078.70968|-214.25903648886197|11292.968716488862|      10.58|   62.93|     0.083|                 95.8|       

...

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|13399.27273| -1340.008688947988|14739.281418947989|      15.29|    78.5|     0.071|                124.2|        110.9| 26140.10764| 16548.26884|    4|   8|
| 12056.8997|-2741.3181544248255|14798.217854424825|      19.43|    89.4|     0.085|                216.0|        171.2| 36677.46171| 25031.95021|   10|  13|
|21767.71084| 1536.7558708649558|20230.954969135044|       16.3|    52.4|     0.081|                0.066|      

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|12178.95812| -2426.308960530976|14605.267080530975|      21.56|    81.2|     4.916|                347.5|         86.0| 33591.50442| 20915.17672|    9|   9|
|28215.56485| 1064.9166342772041|27150.648215722795|      27.17|   30.54|     4.908|                 0.11|        0.082| 31657.67442| 21850.63291|    7|   1|
|16145.97839| -1392.193142812539| 17538.17153281254|      14.29|   68.89|     0.084|                0.033|      

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|          residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|15030.36145|114.10153952265318|14916.259910477347|      14.36|   68.86|     0.077|                0.081|        0.156| 24322.02532|   15643.769|    1|   6|
|13379.20327| 745.3152046829691| 12633.88806531703|      22.04|   65.99|     0.278|                0.062|        0.133| 27793.27434| 17060.70686|    9|   2|
|14365.02024| 741.0347764988455|13623.985463501154|      21.22|    43.2|     0.081|                 0.08|        0.10

-------------------------------------------
Batch: 18
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|8350.843373|-5459.5219076136855|13810.365280613685|      17.11|   69.63|     0.072|                411.8|        56.82| 31587.69231| 26743.38843|   11|  11|
|16823.13253| 1026.6930068850124|15796.439523114987|       15.8|   54.06|     0.075|                449.2|        41.52| 29911.89873| 18036.47416|    1|  14|
|22797.78462| 2679.5627532682556|20118.221866731743|      27.79|    50.4|     4.919|                881.0|      

-------------------------------------------
Batch: 19
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|28631.47335|-2539.8530195740095| 31171.32636957401|      25.85|    76.0|     4.919|                0.069|         0.13| 46303.75139| 28404.64625|    8|  22|
|19755.18072|   933.262145423123|18821.918574576877|      15.49|    76.4|      0.07|                0.048|        0.063| 37698.46154| 31752.89256|   11|  20|
|19093.55649|-4194.7577503024695| 23288.31424030247|      25.71|    74.9|     4.916|                424.5|      

## **Producing the Data**
At this point, the `power_streaming_data.csv` file should be uploaded into the main directory in `jupyterhub`. Additionally, the `Loring_FinalProject.py` file reads in the `power_streaming_data.csv` file and writes a loop that randomly samples some rows from this file, outputs them to a `.csv` (removing the indices), and pauses for 10 seconds in between outputting of datasets.

After running the above line of code to start the query, we can run the loop from the `Loring_FinalProject.py` file in a python console. Once we do this, we will see output here in this notebook right above the start of this section. Then the query will be stopped. Please see the video uploaded to Moodle within the Final Project assignment to see a recording of this process play out in real time.

In [27]:
query.stop()